# 📓 Semana 21 · Dia 4 — Consistência SQL: validando contra o Lakehouse

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | Portfólio empresarial |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Validações SQL no motor |

---


## 📖 Teoria — Validações com contexto do Lakehouse

A camada 3 usa o próprio Lakehouse como fonte da verdade:
- produto existe? (dim_produto)
- cliente já cadastrado? (duplicidade)
- preço dentro da faixa histórica? (ouro)

É o que torna a validação 'inteligente' — não só sintática, mas semântica.


### 💻 Na prática — Validações SQL

Implemente as regras de consistência.


In [ ]:
# Produto existe?
def produto_existe(codigo):
    n = spark.sql(f"SELECT COUNT(*) FROM workspace.prata.dim_produto WHERE StockCode = '{codigo}'").collect()[0][0]
    return n > 0
print("85123A existe:", produto_existe("85123A"))
print("ZZZZZ existe:", produto_existe("ZZZZZ"))

In [ ]:
# Cliente duplicado?
def cliente_duplicado(email):
    n = spark.sql(f"SELECT COUNT(*) FROM workspace.bronze.clientes_bronze WHERE lower(Email) = lower('{email}')").count()
    return n > 0
print("Cliente duplicado? (use um email do seu dataset)")

In [ ]:
# Preço dentro da faixa histórica (Ouro)
def preco_fora_faixa(codigo, preco):
    faixa = spark.sql(f"""
        SELECT MIN(UnitPrice) AS mn, MAX(UnitPrice) AS mx
        FROM workspace.bronze.vendas_bronze WHERE StockCode = '{codigo}'
    """).collect()[0]
    if faixa.mn is None: return False
    return not (faixa.mn * 0.5 <= preco <= faixa.mx * 2.0)
print("preco_fora_faixa implementado (50%–200% do histórico).")

### 💻 Na prática — Integrando ao motor

A camada 3 roda em lote (join com as dimensões) para eficiência.


In [ ]:
# Lote eficiente: join em vez de N queries
print("""
Em vez de validar linha a linha (N queries), faça:
df_submissao.join(dim_produto, on="produto", how="left_anti")
  -> linhas cujo produto NÃO existe (validação em 1 scan)
""")
print("Consistência SQL em lote: rápido e escalável.")

> 🎯 **Dica de prova**: Portfólio: validação com o Lakehouse (anti-join, faixas históricas) é o diferencial — mostra que o app não valida 'no vácuo'.


## 🎯 Exercícios de fixação

**1.** Valide duplicidade de produto no fluxo preços (anti-join).

**2.** Por que validar em lote (join) e não linha a linha?

**3.** Adicione uma regra de faixa ao YAML (min/max dinâmicos do histórico).


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Anti-join

`df.join(dim_produto, 'produto', 'left_anti')` → linhas com produto inexistente.

**2.** Lote

N queries = N scans; join = 1 scan — ordem de grandeza mais rápido.

**3.** Faixa dinâmica

Adicione `regra: faixa_historica` no YAML; a camada 3 resolve com o Ouro.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*